# Credit Risk Scorecard Development
**Objective:** Build a robust, regulatory-grade Application/Behavioral Scorecard to predict probability of default (Target: `femi15plus`).

**Methodology:** Weight of Evidence (WOE) Transformation + Logistic Regression.
**Key Constraints:**
1.  **Interpretability:** Model must be explainable (Points-based system).
2.  **Stability:** Logic must hold over time (Monotonic trends).
3.  **Business Logic:** Binning must align with business intuition (e.g., Higher Savings = Lower Risk).


## Phase 1: Feature Selection & Orthogonality Check
**Goal:** Reduce the initial feature space (~600 variables) to a manageable set of distinct risk drivers.

**Steps Taken:**
1.  **Information Value (IV) Filter:** Calculated IV for all variables using fine binning (20 bins). Dropped weak predictors (IV < 0.02).
2.  **Correlation Clustering:**
    * Selected the top variable from each logical cluster (Delinquency, Liquidity, Spend, Wealth).
    * Applied a hard correlation cap (`corr < 0.6`) to ensure orthogonality.
3.  **Multicollinearity Check (VIF):** Removed variables with Variance Inflation Factor (VIF) > 5 to prevent coefficient instability.
4.  **Significance Testing:** Applied Backward Elimination based on P-values (Threshold > 0.05) to remove statistically insignificant features.

**Outcome:** Reduced variable set from ~600 $\rightarrow$ 30 $\rightarrow$ **Final 7 Key Drivers**.

## Phase 2: Binning Strategy (WOE Transformation)
**Goal:** Convert raw data into linear "Weight of Evidence" (WOE) values to capture non-linear risk patterns.

**Evolution of Binning Logic:**
1.  **Iteration 1 (Statistical):** Used `qcut` (Quantile Binning).
    * *Issue:* Created jagged trends and "V-shapes" (e.g., Low Balance appeared safer than High Balance due to outliers).
2.  **Iteration 2 (Monotonic):** Forced strict monotonicity (Risk must strictly increase or decrease).
    * *Issue:* Created awkward cut-offs (e.g., 1432.55) that are hard to explain to stakeholders.
3.  **Iteration 3 (Business Logic - FINAL):**
    * **Rounding:** Adjusted cuts to logical numbers (e.g., 2500, 5000, 15000).
    * **Missing Values:**
        * For *Investments* (EPF): Treated "Missing" as "0" (No Investment).
        * For *Credit History* (Loan Negatives): Treated "Missing" as distinct (Unknown Risk vs Clean History).
    * **Concentration Fix:** Split massive bins (e.g., Closing Balance < 2500) to ensure differentiation between "Negative Balance" and "Low Positive Balance".


In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from scipy.stats import ks_2samp

# ==========================================
# 1. CONFIGURATION
# ==========================================
# FINAL CLEAN LIST (7 Variables)
SELECTED_FEATURES = [
    'var206050', 'var206027', 'var202032', 'var203001', 
    'var303018', 'var201093', 'var201793'
]

# Update path if necessary
FILE_PATH = '20k_base_with_independent_variables.csv'
VAR_NAMES_PATH = 'var_names.csv'
TARGET_COL = 'femi15plus'

PDO = 20
BASE_SCORE = 600
BASE_ODDS = 50

# ==========================================
# 2. REFINED BINNING
# ==========================================
CUSTOM_BINS = {
    'var203001': [-np.inf, 0, 5000, 20000, 50000, np.inf],
    'var201093': [-np.inf, 15000, np.inf], 
    'var201793': [-np.inf, 2500, 25000, np.inf], 
    'var202032': [-np.inf, 1000, 6000, 10000, np.inf],
    'var303018': [-np.inf, 1000, 3000, 6000, 10000, np.inf],
    'var206050': [-np.inf, 0, 1, 2, np.inf],
    'var206027': [-np.inf, 0, 5000, 15000, np.inf]
}

GROUP_MISSING_WITH_ZERO = ['var203001', 'var202032'] 

# ==========================================
# 3. WOE TRANSFORMER CLASS
# ==========================================
class WOE_Transformer:
    def __init__(self):
        self.woe_maps = {}
        self.pop_maps = {} 

    def fit(self, X, y):
        df = X.copy()
        df['target'] = y
        for col in X.columns:
            if col in GROUP_MISSING_WITH_ZERO:
                df[col] = df[col].fillna(0)
                missing_mask = pd.Series(False, index=df.index)
            else:
                missing_mask = df[col].isnull()

            bins = CUSTOM_BINS.get(col, [-np.inf, np.inf])
            df[f'{col}_bin'] = pd.cut(df[col], bins=bins).astype(str)
            if not missing_mask.empty and missing_mask.any():
                df.loc[missing_mask, f'{col}_bin'] = 'Missing'
            self._calc_woe(df, col, f'{col}_bin')

    def _calc_woe(self, df, col, bin_col):
        stats = df.groupby(bin_col)['target'].agg(['count', 'sum'])
        stats['good'] = stats['count'] - stats['sum']
        stats['bad'] = stats['sum']
        g_tot = max(stats['good'].sum(), 1)
        b_tot = max(stats['bad'].sum(), 1)
        total_n = stats['count'].sum()
        stats['woe'] = np.log((stats['good']/g_tot + 0.0001) / (stats['bad']/b_tot + 0.0001))
        stats['pop_pct'] = stats['count'] / total_n
        self.woe_maps[col] = stats['woe'].to_dict()
        self.pop_maps[col] = stats['pop_pct'].to_dict()

    def transform(self, X):
        X_woe = pd.DataFrame(index=X.index)
        for col in X.columns:
            if col not in self.woe_maps: continue
            series = X[col].copy()
            mapping = self.woe_maps[col]
            bins = CUSTOM_BINS.get(col)
            if col in GROUP_MISSING_WITH_ZERO:
                series = series.fillna(0)
                is_missing = pd.Series(False, index=series.index)
            else:
                is_missing = series.isnull()
            cuts = pd.cut(series, bins=bins).astype(str)
            woe_series = cuts.map(mapping)
            if mapping.get('Missing') is not None:
                woe_series[is_missing] = mapping['Missing']
            X_woe[col] = woe_series.fillna(0).astype(float)
        return X_woe

## Phase 3: Model Training & Validation
**Algorithm:** Logistic Regression (L1 Penalty for stability).

**Performance Evolution:**
* **XGBoost Baseline:** High Train AUC (0.96) but massive overfitting (Test AUC 0.67).
* **Raw Logistic Regression:** Poor separation (KS ~19%) due to inability to handle non-linearity.
* **WOE Logistic Regression (Final):**
    * **Train AUC:** ~0.71
    * **Test AUC:** ~0.69
    * **Stability Gap:** < 3% (Excellent stability)
    * **Test KS:** ~28% (Strong commercial separation)

**Rank Ordering:**
The final model demonstrates clean rank ordering across deciles.
* **Decile 10 (High Risk):** ~5-6% Bad Rate
* **Decile 1 (Low Risk):** < 0.5% Bad Rate
* *Correction Note:* Fixed the inversion between Decile 5 and 6 by refining the Balance variable binning.

In [ ]:
def calculate_ks(y_true, y_prob):
    return ks_2samp(y_prob[y_true == 1], y_prob[y_true == 0]).statistic

def create_decile_table(y_true, y_prob, name="Dataset"):
    df = pd.DataFrame({'target': y_true, 'prob': y_prob})
    df['decile'] = pd.qcut(df['prob'], 10, labels=False, duplicates='drop')
    stats = df.groupby('decile').agg(
        Count=('target', 'count'), Bad_Rate=('target', 'mean'),
        Min_PD=('prob', 'min'), Avg_PD=('prob', 'mean'), Max_PD=('prob', 'max'),
        Bads=('target', 'sum')
    ).sort_index(ascending=False)
    stats['Goods'] = stats['Count'] - stats['Bads']
    stats['Cum_Bad'] = stats['Bads'].cumsum()
    stats['Cum_Good'] = stats['Goods'].cumsum()
    stats['Cum_Bad_Pct'] = stats['Cum_Bad'] / stats['Bads'].sum()
    stats['Cum_Good_Pct'] = stats['Cum_Good'] / stats['Goods'].sum()
    stats['KS'] = np.abs(stats['Cum_Bad_Pct'] - stats['Cum_Good_Pct']) * 100
    stats = stats.reset_index()
    stats['decile'] = stats['decile'] + 1
    print(f"\n=== {name} DECILE TABLE ===")
    cols = ['decile', 'Count', 'Bad_Rate', 'Min_PD', 'Avg_PD', 'Max_PD', 'Cum_Bad', 'Cum_Good', 'KS']
    print(stats[cols].to_string(index=False, float_format="{:.4f}".format))

print("Loading Data...")
try:
    df = pd.read_csv(FILE_PATH, low_memory=False)
    df = df.dropna(subset=[TARGET_COL])
    X = df[SELECTED_FEATURES]
    y = df[TARGET_COL]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
    
    # 1. Transform
    print("Transforming to WOE...")
    transformer = WOE_Transformer()
    transformer.fit(X_train, y_train)
    X_train_woe = transformer.transform(X_train)
    X_test_woe = transformer.transform(X_test)
    
    # 2. Final Model
    print("Training Final Model...")
    final_model = sm.Logit(y_train, sm.add_constant(X_train_woe)).fit(disp=0)
    
    train_probs = final_model.predict(sm.add_constant(X_train_woe))
    test_probs = final_model.predict(sm.add_constant(X_test_woe))
    
    print(f"\nTRAIN AUC: {roc_auc_score(y_train, train_probs):.4f}")
    print(f"TEST  AUC: {roc_auc_score(y_test, test_probs):.4f}")
    
    create_decile_table(y_train, train_probs, "TRAIN")
    create_decile_table(y_test, test_probs, "TEST")

except Exception as e:
    print(f"Error: {e}")

## Phase 4: Final Scorecard Logic
The model consists of **7 Variables** covering 4 distinct risk dimensions.

| Variable | Description | Risk Dimension | Logic |
| :--- | :--- | :--- | :--- |
| **var206050** | Loan Negative Events (6M) | **Delinquency** | 0 events is good. >1 is high risk. |
| **var206027** | Missed Payment Amt (6M) | **Severity** | Higher missed amount = Higher Risk. |
| **var202032** | Avg Credit Card Spend (12M) | **Spend/Lifestyle** | Active usage (>1k) indicates capacity. Missing/Inactive is riskier. |
| **var303018** | Utility Payments (12M) | **Stability** | Regular utility payments indicate residential stability. |
| **var203001** | EPF Investment | **Wealth** | Presence of EPF is a strong safety signal. |
| **var201093** | Avg Balance (Lifetime) | **Long-term Wealth** | Higher average balance (>15k) gets max points. |
| **var201793** | Closing Balance (1M) | **Liquidity** | Negative/Zero balance is high risk. >25k is very safe. |

**Scaling:**
* **Base Score:** 600 points @ 50:1 Odds
* **PDO:** 20 points (Score increases by 20 when odds double)

In [ ]:
def get_lower_bound(bin_str):
    if str(bin_str) == 'Missing': return -999999999.0
    try:
        clean = bin_str.replace('(','').replace('[','').replace(']','').split(',')[0]
        return float(clean)
    except: return 0.0

# 3. Scorecard Export
try:
    intercept = final_model.params['const']
    coeffs = final_model.params.drop('const')
    factor = PDO / np.log(2)
    offset = BASE_SCORE - (factor * np.log(BASE_ODDS))
    
    sc_data = []
    for var in SELECTED_FEATURES:
        beta = coeffs[var]
        mapping = transformer.woe_maps[var]
        pop_mapping = transformer.pop_maps[var]
        for bin_lbl, woe_val in mapping.items():
            pts = - (woe_val * beta + intercept/len(SELECTED_FEATURES)) * factor + (offset / len(SELECTED_FEATURES))
            pop_pct = pop_mapping.get(bin_lbl, 0.0)
            sc_data.append({
                'Variable': var, 'Bin': bin_lbl, 
                'Population %': f"{pop_pct:.1%}", 'WOE': round(woe_val, 4), 
                'Points': int(pts), 'Lower_Bound': get_lower_bound(bin_lbl)
            })
    
    sc_df = pd.DataFrame(sc_data)
    try:
        var_desc = pd.read_csv(VAR_NAMES_PATH)
        if 'variable' in var_desc.columns: var_desc.rename(columns={'variable': 'Variable'}, inplace=True)
        sc_df = pd.merge(sc_df, var_desc[['Variable', 'Description']], on='Variable', how='left')
    except: sc_df['Description'] = ""
    
    sc_df = sc_df.sort_values(by=['Variable', 'Lower_Bound'])
    sc_df = sc_df[['Variable', 'Description', 'Bin', 'Population %', 'WOE', 'Points']]
    
    print("\n=== FINAL SCORECARD (Sample) ===")
    print(sc_df.head(10).to_string(index=False))
    sc_df.to_csv("Final_Scorecard_Documented.csv", index=False)
    print("\nSuccess! Saved to 'Final_Scorecard_Documented.csv'")

except Exception as e:
    print(f"Error generating scorecard: {e}")